# Plot results for offline prompting in the "dragon" example

In [1]:
%cd ..
%pwd  # should be "llm-adaptation"

C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation


C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation\.venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


'C:\\Users\\micha\\OneDrive - Univerzita Karlova\\research\\2024-LLM-DEECo\\llm-adaptation'

In [2]:
from pathlib import Path
import pandas as pd
import shutil
import subprocess
import sys
import json

In [3]:
results_folder = Path("generated_adaptations/dragon")

## Run experiments

In [16]:
prompt = "generated_adaptations/prompts/dragon_strategy.md"
variants = ["default", "notest", "constraints"]
llms = {
    "5nano": "gpt-5-nano-2025-08-07",
}
repeats = 1
start = 1

In [25]:
# import generated_adaptations.generator as generator

In [17]:
for variant in variants:
    for llm_name, llm in llms.items():
        for repeat in range(start, start + repeats):
            folder_name = f"{llm_name}_{repeat:02d}"
            print(f"\n{variant}/{folder_name}\n")
            folder = results_folder / variant / folder_name
            folder.mkdir(parents=True, exist_ok=True)
            shutil.copy(prompt, folder / "01_01_user.md")
            cmd  = [sys.executable, "generated_adaptations/generator.py", f"--folder={str(folder)}", f"--llm={llm}"]  #, "--retries_test=1", "--retries_simulation=0"]
            result = subprocess.run(cmd, capture_output=True, text=True)  # even capture_output=False does not stream in real-time, so we rather capture it and print it later
            print(result.stdout)
            print(result.stderr)
            # generator.main([f"--folder={str(folder)}"])  # this also does not show the real-time output


default/5nano_01

Loaded 2 messages from generated_adaptations\dragon\default\5nano_01.
Querying LLM with prompt 01_01_user.
LLM response saved to '01_02_llm.md'.
Code block saved to 'generated_adaptations\dragon\default\5nano_01\code_01_02.py'.
TOKENS USED:
Input: 1066, Output: 7969 (reasoning: 6976)
Response time (seconds): 51.2

Running tests: pytest generated_adaptations/tests -q --tb=short -rfExXpP --show-capture=no --color=no --example=dragon --adaptation_name=5nano_01/code_01_02 --variant=default
Test exit code: 1
Running simulation for 'generated_adaptations\dragon\default\5nano_01\code_01_02.py'.
  Run #1/3: python main.py dragon/configs/default.yaml generated_adaptations/configs/generated.yaml DSL/dragon.yaml --extra_config={"name": "default/5nano_01/code_01_02", "log_dir.append": "/default/5nano_01/code_01_02", "adaptation_name": "generated_adaptations.dragon.default.5nano_01.code_01_02.SmartAdaptation"} -s 1 -e 1
    StdErr: 60 lines
    In 'assign_in_village': Component a

## Results

In [18]:
folders = list(results_folder.glob("*/5nano_*"))
# folders = [results_folder / "41mini"]
print([(f.parent.stem, f.stem) for f in folders])

[('constraints', '5nano_01'), ('default', '5nano_01'), ('notest', '5nano_01')]


In [19]:
summary = pd.DataFrame(columns=["llm", "params", "repeat"])
best = pd.DataFrame(columns=["llm", "params", "repeat"])
for folder in folders:
    llm, repeat = folder.stem.split("_")
    params = folder.parent.stem
    summary.loc[len(summary), ["llm", "params", "repeat"]] = [llm, params, repeat]
    best.loc[len(best), ["llm", "params", "repeat"]] = [llm, params, repeat]
    for file in (folder / "results").glob("*.txt"):
        name = file.stem.removeprefix("code_")
        if "_test_fail" in name:
            code = name.removesuffix("_test_fail")
            summary.loc[len(summary) - 1, code + "_test"] = "fail"
        elif "_test_pass" in name:
            code = name.removesuffix("_test_pass")
            summary.loc[len(summary) - 1, code + "_test"] = "pass"
        else:
            print(f"Unknown file: {file}")
    for file in (folder / "results").glob("*.json"):
        name = file.stem.removeprefix("code_")
        if "_simulation_result" in name:
            code = name.removesuffix("_simulation_result")
            results = json.load(open(file))
            result = results["winrate"]
            summary.loc[len(summary) - 1, code + "_result"] = result
            best.loc[len(best) - 1, code.split("_")[0] + "_result"] = result  # only take the first part of the code file name
        else:
            print(f"Unknown file: {file}")
summary = summary.astype("object")
best = best.astype("object")
summary.fillna("", inplace=True)
best.fillna("", inplace=True)

In [20]:
summary.set_index(["llm", "params", "repeat"], inplace=True)
best.set_index(["llm", "params", "repeat"], inplace=True)

In [21]:
summary = summary.reindex(sorted(summary.columns), axis=1)
best = best.reindex(sorted(best.columns), axis=1)

In [22]:
summary

01_02_result 01_02_test 01_04_result 01_04_test  \
llm   params      repeat                                                   
5nano constraints 01              1.0       fail          0.0       fail   
      default     01              0.0       fail          0.0       pass   
      notest      01              1.0       pass                           

                         01_06_result 01_06_test 01_08_result 01_08_test  \
llm   params      repeat                                                   
5nano constraints 01              0.0       fail          1.0       fail   
      default     01                                                       
      notest      01                                                       

                         02_02_result 02_02_test  ... 02_08_result 02_08_test  \
llm   params      repeat                          ...                           
5nano constraints 01              0.0       fail  ...          1.0       fail   
      default     01              0.0       pass  ...                           
      notest      01              0.0       pass  ...                           

                         03_02_result 03_02_test 03_04_result 03_04_test  \
llm   params      repeat                                                   
5nano constraints 01              1.0       fail          0.0       fail   
      default     01              0.0       pass                           
      notest      01              1.0       pass                           

                         03_06_result 03_06_test 03_08_result 03_08_test  
llm   params      repeat                                                  
5nano constraints 01              1.0       fail          0.0       fail  
      default     01                                                      
      notest      01                                                      

[3 rows x 24 columns]

In [23]:
best

01_result 02_result 03_result
llm   params      repeat                              
5nano constraints 01           1.0       1.0       0.0
      default     01           0.0       0.0       0.0
      notest      01           1.0       0.0       1.0